In [ ]:
## Overview

This notebook implements the online query service of a GDPR Q&A system through a LINE Bot interface.

This project reflects an exploration of AI-assisted compliance tools, focusing on how semantic retrieval can be applied to privacy law knowledge management.

Instead of generating answers in real time using a large language model, this system retrieves responses from a pre-validated QA dataset.

This design ensures:
- consistency of legal answers
- traceability to source regulations
- reduced risk of hallucination in compliance scenarios

When a user submits a question via LINE Bot:

1. The query is converted into an embedding vector
2. Cosine similarity is used to compare it with precomputed QA embeddings
3. The most relevant validated answer is retrieved
4. The system returns the response directly without real-time LLM generation

This design reduces the risk of legal hallucination and ensures that all responses are derived from a controlled and reviewed knowledge base.

Note:
- Outputs are removed for security and readability
- API keys are managed via environment variables and are not included in this notebook

In [ ]:
## API Key Setup

This notebook requires an OpenAI API key.

Before running, please set your API key as an environment variable:

```python
import os
os.environ["OPENAI_API_KEY"] = "your-api-key"

In [ ]:
## Security Note

Sensitive credentials (API keys, tokens) are managed via environment variables
and are not included in this repository.

In [ ]:
## Design Note

# This system follows a retrieval-based architecture instead of real-time LLM generation.
# All responses are served from a pre-validated QA knowledge base to ensure:
# - consistency of legal answers
# - traceability to source regulations
# - reduced hallucination risk in legal contexts

In [ ]:
# Install required packages for LINE Bot and semantic retrieval
!pip install flask line-bot-sdk pyngrok deep-translator openai

In [ ]:
# =========================
# Development Server Exposure (ngrok)
# =========================

# Expose the local Flask server to a public URL for LINE webhook integration
# during development and testing

import os
from pyngrok import ngrok

NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(5000).public_url
print("Webhook URL:", public_url)

In [ ]:
from flask import Flask, request, abort
from linebot import LineBotApi, WebhookHandler
from linebot.models import MessageEvent, TextMessage, TextSendMessage
from linebot.exceptions import InvalidSignatureError
import os
import json

In [ ]:
LINE_CHANNEL_ACCESS_TOKEN = os.getenv("LINE_CHANNEL_ACCESS_TOKEN")
LINE_CHANNEL_SECRET = os.getenv("LINE_CHANNEL_SECRET")
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# =========================
# Load QA Knowledge Base
# =========================

# Load finalized QA dataset and precomputed embeddings
# These serve as the retrieval source for all user queries

# Design Note:
# Precomputed embeddings are used to:
# - reduce API latency and cost
# - enable fast similarity search at runtime
# - ensure consistent retrieval performance

with open("qa_v1_final.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

with open("qa_v1_embeddings.json", "r", encoding="utf-8") as f:
    qa_embeddings = json.load(f)

print("Number of QA pairs:", len(qa_data))
print("Number of embeddings:", len(qa_embeddings))


In [ ]:
import re
import numpy as np
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Normalize user input to improve retrieval consistency
# This step removes formatting noise and standardizes text input
def normalize_question(q: str) -> str:
    q = q.strip()
    q = q.replace("？", "?")
    q = re.sub(r"\s+", " ", q)
    return q

# Compute cosine similarity between query embedding and QA embeddings
# Used to identify the most relevant pre-validated answer
def cosine_sim(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

from deep_translator import GoogleTranslator
translator = GoogleTranslator(source="auto", target="zh-TW")

def find_gdpr_answer(user_question: str, threshold=0.7):
    user_question_norm = normalize_question(user_question)

    # Detect non-Chinese queries and translate them into Chinese
    # to align with the language of the QA knowledge base

    # Design Note:
    # The QA knowledge base is constructed in Chinese.
    # Translating queries ensures consistent embedding space and improves retrieval accuracy.
    if re.search(r"[a-zA-Z]", user_question_norm):
        try:
            user_question_norm = translator.translate(user_question_norm)
            user_question_norm = normalize_question(user_question_norm)
        except Exception as e:
            print("Translation failed:", e)

    # Generate embedding for the normalized user query
    # This embedding is used for semantic similarity matching
    resp = client.embeddings.create(
        model="text-embedding-3-small",
        input=user_question_norm
    )
    user_emb = resp.data[0].embedding


    # Perform similarity search against all stored QA embeddings
    # Select the question with the highest semantic similarity score

    # Design Note:
    # A simple cosine similarity search is used for transparency and interpretability,
    # allowing clear traceability from user query to matched QA entry.
    best_q = None
    best_score = -1

    for q, emb in qa_embeddings.items():
        score = cosine_sim(user_emb, emb)
        if score > best_score:
            best_score = score
            best_q = q

    # Fallback response when no relevant QA match is found
    # Triggered when similarity score is below the defined threshold

    # Design Note:
    # Instead of forcing a low-confidence answer, the system returns a safe fallback response
    # to avoid incorrect legal guidance.
    if best_score < threshold:
        return (
        "本機器人目前僅提供 GDPR 相關法規問答，請重新提問。\n\n"
        "This bot currently supports GDPR-related legal Q&A only. Please rephrase your question."
    )

    # Return the pre-validated answer associated with the best-matching question
    answer_list = qa_data[best_q]
    return "\n".join(answer_list)


In [ ]:

# =========================
# LINE Bot Initialization
# =========================

# Initialize Flask app and LINE Messaging API client

import os
from flask import Flask
from linebot import LineBotApi, WebhookHandler

app = Flask(__name__)

LINE_CHANNEL_ACCESS_TOKEN = os.getenv("LINE_CHANNEL_ACCESS_TOKEN")
LINE_CHANNEL_SECRET = os.getenv("LINE_CHANNEL_SECRET")

if not LINE_CHANNEL_ACCESS_TOKEN or not LINE_CHANNEL_SECRET:
    raise ValueError("Missing LINE channel credentials. Please set environment variables.")

line_bot_api = LineBotApi(LINE_CHANNEL_ACCESS_TOKEN)
handler = WebhookHandler(LINE_CHANNEL_SECRET)

# =========================
# LINE Webhook Endpoint
# =========================

# Handle incoming requests from LINE platform
# Verify request signature before processing events
@app.route("/callback", methods=["POST"])
def callback():
    signature = request.headers.get("X-Line-Signature", "")
    body = request.get_data(as_text=True)

    # Some verification requests may not include a signature
    # Return OK directly for such cases
    if not signature:
        return "OK"

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        abort(400)

    return "OK"

# =========================
# Message Handling Logic
# =========================

# Receive user message, perform retrieval, and return the matched answer

# Design Note:
# The handler strictly uses retrieval results and does not invoke LLM generation,
# ensuring deterministic and auditable responses.
@handler.add(MessageEvent, message=TextMessage)
def handle_message(event):
    try:
        user_text = event.message.text
        reply_text = find_gdpr_answer(user_text)

        line_bot_api.reply_message(
            event.reply_token,
            TextSendMessage(text=reply_text)
        )

    # Handle unexpected errors and return a safe fallback message to the user
    except Exception as e:
        print("handle_message error:", e)
        line_bot_api.reply_message(
            event.reply_token,
            TextSendMessage(text="An error occurred. Please try again later.（系統發生錯誤，請稍後再試）")
        )

# Start the Flask application (development mode)
if __name__ == "__main__":
    app.run(port=5000)
